In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import re
import datetime
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import pickle

In [2]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番', '展示', '3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻']

In [3]:
k_files = glob.glob("../csv/K*")
b_files = glob.glob("../csv/B*")

In [4]:
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp

In [5]:
print("Concat K-files")
kdf = concat_files(k_files)
print("Concat B-files")
bdf = concat_files(b_files)

Concat K-files


100%|██████████| 1426/1426 [04:25<00:00,  5.38it/s]


Concat B-files


100%|██████████| 1396/1396 [02:12<00:00, 10.54it/s]


In [6]:
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 913512 entries, 0 to 913511
Data columns (total 26 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   艇番       913512 non-null  int64  
 1   選手登番     913512 non-null  object 
 2   選手名      913512 non-null  object 
 3   年齢       913512 non-null  int64  
 4   支部       913512 non-null  object 
 5   体重       913512 non-null  int64  
 6   級別       913512 non-null  object 
 7   全国勝率     913512 non-null  float64
 8   全国2連率    913512 non-null  float64
 9   当地勝率     913512 non-null  float64
 10  当地2連率    913512 non-null  float64
 11  モーター2連率  913512 non-null  float64
 12  ボート2連率   913512 non-null  float64
 13  RaceID   913512 non-null  object 
 14  場所       913512 non-null  int64  
 15  R        913512 non-null  object 
 16  着        913512 non-null  object 
 17  展示       913512 non-null  object 
 18  3連単_結果   913512 non-null  object 
 19  3連単_払戻   913512 non-null  object 
 20  3連複_結果   913512 non-null  

In [8]:
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid


In [9]:
df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)

In [10]:
zero = []
for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero

100%|██████████| 913512/913512 [00:05<00:00, 153804.09it/s]


In [11]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 913512 entries, 0 to 913511
Data columns (total 26 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   艇番       913512 non-null  int64  
 1   選手登番     913512 non-null  object 
 2   選手名      913512 non-null  object 
 3   年齢       913512 non-null  int64  
 4   支部       913512 non-null  int64  
 5   体重       913512 non-null  int64  
 6   級別       913512 non-null  int64  
 7   全国勝率     913512 non-null  float64
 8   全国2連率    913512 non-null  float64
 9   当地勝率     913512 non-null  float64
 10  当地2連率    913512 non-null  float64
 11  モーター2連率  913512 non-null  float64
 12  ボート2連率   913512 non-null  float64
 13  RaceID   913512 non-null  object 
 14  場所       913512 non-null  int64  
 15  R        913512 non-null  int64  
 16  着        913512 non-null  object 
 17  展示       913512 non-null  object 
 18  3連単_結果   913512 non-null  object 
 19  3連単_払戻   913512 non-null  object 
 20  3連複_結果   913512 non-null  

In [12]:
os.makedirs('../train_csv',exist_ok=True)
df_encoded.to_csv('../train_csv/train.csv')